<div align="center">
  <h3><b>ESCUELA POLITÉCNICA NACIONAL</b></h3>
  <h3><b>FACULTAD DE SISTEMAS</b></h3>
  <h3><b>INGENIERÍA EN CIENCIAS DE LA COMPUTACIÓN</b></h3>
  <h3><b>RECUPERACIÓN DE LA INFORMACIÓN</b></h3>
</div>

---
**Nombre**   Mark Hernández       
**Fecha**    27/05/26  
**Docente**  Iván Carrera

#### Librerías:

Cargamos las librerías del modelo de embeddings

In [1]:
%pip install -q sentence-transformers

Note: you may need to restart the kernel to use updated packages.


Cargamos las librerías necesarias:

In [2]:
import kagglehub
import os
import pandas as pd
import re
import unicodedata
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np
import pickle

c:\Users\mark_\Documents\ir26a\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Examen Primer Bimestre 

#### 1. Carga del Corpus

Primero realizamos la carga del corpus para el dataset **Rotten Tomatoes movies and critic reviews dataset** desde Kagglehub.

In [3]:
# Download latest version
path = kagglehub.dataset_download("stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset")

print("Path to dataset files:", path)

print("Archivos en la carpetea descargada:")
for f in os.listdir(path):
    print(f)

Path to dataset files: C:\Users\mark_\.cache\kagglehub\datasets\stefanoleone992\rotten-tomatoes-movies-and-critic-reviews-dataset\versions\1
Archivos en la carpetea descargada:
rotten_tomatoes_critic_reviews.csv
rotten_tomatoes_movies.csv


Antes de escoger con que archivo trabajar revisamos que campos posee cada uno.

In [4]:
# Revisamos los campos de cada CSV
for f in os.listdir(path):
    filepath = os.path.join(path, f)
    df = pd.read_csv(filepath, nrows=0)  # Leemos el header sin cargar datos.
    print(f"\n📄 {f}")
    print(f"   Campos ({len(df.columns)}):")
    for col in df.columns:
        print(f"     - {col}")


📄 rotten_tomatoes_critic_reviews.csv
   Campos (8):
     - rotten_tomatoes_link
     - critic_name
     - top_critic
     - publisher_name
     - review_type
     - review_score
     - review_date
     - review_content

📄 rotten_tomatoes_movies.csv
   Campos (22):
     - rotten_tomatoes_link
     - movie_title
     - movie_info
     - critics_consensus
     - content_rating
     - genres
     - directors
     - authors
     - actors
     - original_release_date
     - streaming_release_date
     - runtime
     - production_company
     - tomatometer_status
     - tomatometer_rating
     - tomatometer_count
     - audience_status
     - audience_rating
     - audience_count
     - tomatometer_top_critics_count
     - tomatometer_fresh_critics_count
     - tomatometer_rotten_critics_count


Luego de revisar los campos, podemos concluir que para este examen se hará uso de ambos archivos. Esta decisión se la toma a partir de analizar los requerimientos para la visualización de las consultas, en la cual se nos pide incluir el título de la película (movie_title) el cual está en el archivo **rotten_tomatoes_movies.csv** y el texto de la crítica (review_content) que se encuentra en el archivo **rotten_tomatoes_critic_reviews.csv**

In [5]:
# --- 1. CARGA Y UNIÓN DE DATASETS ---
print("Cargando datasets...")
movies_path = f"{path}/rotten_tomatoes_movies.csv"
reviews_path = f"{path}/rotten_tomatoes_critic_reviews.csv"

movies_df = pd.read_csv(movies_path)
reviews_df = pd.read_csv(reviews_path)

# Unimos utilizando 'rotten_tomatoes_link' (realizamos un inner join para conservar solo reseñas con película asociada)
print("Uniendo datasets y seleccionando campos...")
df_merged = pd.merge(reviews_df, movies_df[['rotten_tomatoes_link', 'movie_title', 'genres']], 
                     on='rotten_tomatoes_link', 
                     how='inner')

# Seleccionamos los campos necesarios: Título, Género, Texto de la reseña y Nombre del crítico.
# Eliminamos las filas sin texto con dropna() y creamos Document ID en lugar del link.
df = df_merged[['movie_title','genres','review_content','critic_name']].dropna(subset=['review_content']).reset_index(drop=True)
df.insert(0, 'Document ID', df.index.map(lambda i: f"Movie_{i}"))

df.head(5)

Cargando datasets...
Uniendo datasets y seleccionando campos...


,Document ID,movie_title,genres,review_content,critic_name
0,Movie_0,Percy Jackson & the Olympians: The Lightning T...,"Action & Adventure, Comedy, Drama, Science Fic...",A fantasy adventure that fuses Greek mythology...,Andrew L. Urban
1,Movie_1,Percy Jackson & the Olympians: The Lightning T...,"Action & Adventure, Comedy, Drama, Science Fic...","Uma Thurman as Medusa, the gorgon with a coiff...",Louise Keller
2,Movie_2,Percy Jackson & the Olympians: The Lightning T...,"Action & Adventure, Comedy, Drama, Science Fic...",With a top-notch cast and dazzling special eff...,NaN
3,Movie_3,Percy Jackson & the Olympians: The Lightning T...,"Action & Adventure, Comedy, Drama, Science Fic...",Whether audiences will get behind The Lightnin...,Ben McEachen
4,Movie_4,Percy Jackson & the Olympians: The Lightning T...,"Action & Adventure, Comedy, Drama, Science Fic...",What's really lacking in The Lightning Thief i...,Ethan Alter


Antes de proceder al preprocesamiento verificamos el tamaño de nuestro dataset.

In [6]:
print(f"Tamaño del corpus: {len(df)}")

Tamaño del corpus: 1064109


Puesto que el tamaño del corpus posee más de 1 millón de documentos, tomaremos una muestra representativa de 20000 documentos.

In [10]:
df = df.sample(n=20000, random_state=42).reset_index(drop=True)
df.head()

,Document ID,movie_title,genres,review_content,critic_name,processed_review
0,Movie_448868,Julie & Julia,"Comedy, Drama",Julie & Julia was directed by Nora Ephron with...,Joe Williams,julie julia was directed by nora ephron with a...
1,Movie_681960,The Road to Love,"Art House & International, Drama","Doesn't add much to the coming-out genre, as i...",Dave Kehr,doesnt add much to the comingout genre as it h...
2,Movie_66002,The Boys Are Back,Drama,Hicks's hand with these relationships is more ...,Tim Robey,hickss hand with these relationships is more t...
3,Movie_314166,Fat Man and Little Boy,Drama,"An interesting historical drama, but lacks the...",Bob Bloom,an interesting historical drama but lacks the ...
4,Movie_357075,Gone Girl,"Drama, Mystery & Suspense","Unfortunately, and more to the point than the ...",Anna Storm,unfortunately and more to the point than the f...


#### 2. Preprocesamiento

Primero creamos la función de limpieza solicitada con conversión de minúsculas, eliminación de signos de puntuación y eliminación de espacios redundantes. Además, se incorporarán 2 técnicas: eliminación de stopwords y normalización de caracteres.

In [11]:
def preprocess_text(texto):
        """
        Convierte a minúsculas, elimina signos de puntuación, elimina espacios redundantes, y aplica la normalización de caracteres.
        """
        if not isinstance(texto, str):
            return []
        
        # 1. Normalización de caracteres (eliminar acentos/diacríticos y pasar a ASCII)
        text = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
        # 2. Conversión a minúsculas
        text = text.lower()
        # 3. Eliminación de signos de puntuación
        text = re.sub(r'[^\w\s]', '', text)
        # 4. Eliminación de stopwords y 5. Eliminación de espacios redundantes
        words = text.split()
        
        return ' '.join(words)

Aplicamos nuestra función de preprocesamiento a la columna *content_review* en el dataframe y creamos una nueva columna llamada *processed_review*.

In [12]:
print("Iniciando preprocesamiento...")

# Aplicar la función al contenido de las reseñas
df['processed_review'] = df['review_content'].apply(preprocess_text)
# Mostrar una muestra para verificar los cambios
df[['review_content', 'processed_review']].head()

Iniciando preprocesamiento...


,review_content,processed_review
0,Julie & Julia was directed by Nora Ephron with...,julie julia was directed by nora ephron with a...
1,"Doesn't add much to the coming-out genre, as i...",doesnt add much to the comingout genre as it h...
2,Hicks's hand with these relationships is more ...,hickss hand with these relationships is more t...
3,"An interesting historical drama, but lacks the...",an interesting historical drama but lacks the ...
4,"Unfortunately, and more to the point than the ...",unfortunately and more to the point than the f...


#### 3. Generación de embeddings

Para la generación del embedding usaremos el modelo *all-MiniLM-L6-v2*. Este modelo genera vectores de 384 dimensiones y, además, es rápido y precios para NLP en inglés.

In [ ]:
# Inicializamos tqdm para pandas (esto nos permite usar progress_apply)
tqdm.pandas()

print("Cargando el modelo de Sentence Transformers...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Ruta para guardar los embeddings y no tener que recalcularlos si se reinicia el kernel
embeddings_file = "review_embeddings.pkl"

if os.path.exists(embeddings_file):
    print("Cargando embeddings previamente guardados...")
    with open(embeddings_file, "rb") as f:
        document_embeddings = pickle.load(f)
    print(f"Se cargaron {len(document_embeddings)} embeddings.")
else:
    print("Generando embeddings...")
    
    # Convertimos la columna de texto preprocesado a una lista
    sentences = df['processed_review'].tolist()
    
    # Generamos los embeddings. show_progress_bar=True es útil para monitorear el avance.
    document_embeddings = model.encode(sentences, show_progress_bar=True, batch_size=32)
    
    # Guardamos los embeddings en disco para futuras ejecuciones
    with open(embeddings_file, "wb") as f:
        pickle.dump(document_embeddings, f)
    print("Embeddings generados y guardados con éxito.")

# Opcional: Verificar las dimensiones del resultado
print(f"Forma de la matriz de embeddings: {document_embeddings.shape}")

Cargando el modelo de Sentence Transformers...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7689.81it/s]


Generando embeddings (esto puede tomar varios minutos dependiendo del tamaño del corpus)...


Batches: 100%|██████████| 625/625 [02:41<00:00,  3.86it/s]


Embeddings generados y guardados con éxito.
Forma de la matriz de embeddings: (20000, 384)


#### 4. Implementación de una función de búsqueda con similitud coseno